####Using Transformer LLMs for Chain of Throught prompting

1. Create a pipeline for text-generation using a model Qwen/Qwen2.5-1.5B-Instruct

In [0]:
from transformers import pipeline, set_seed
pipe = pipeline(task="text-generation", model="Qwen/Qwen2.5-1.5B-Instruct")
set_seed(45)

2026-03-09 07:00:23.013387: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-09 07:00:23.026542: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2026-03-09 07:00:23.117137: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2026-03-09 07:00:23.181193: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773039623.201841    1544 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773039623.20

[2026-03-09 07:00:31,432] [WARNING] [real_accelerator.py:194:get_accelerator] Setting accelerator to CPU. If you have GPU or other accelerator, we were unable to detect it.
[2026-03-09 07:00:31,438] [INFO] [real_accelerator.py:239:get_accelerator] Setting ds_accelerator to cpu (auto detect)


/usr/bin/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


...

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Device set to use cpu


2. Create a prompt for a problem solving which could automatically trigger chain of thought solution.

In [0]:
prompt="""
I went to the market and bought 10 apples. I gave 2 apples to the neighbor and 2 to the repairman. I then went and bought 5 more apples and ate 1. How many apples did I remain with?
"""
result = pipe(prompt, min_new_tokens=5, max_new_tokens=200)
print(result[0]["generated_text"])


I went to the market and bought 10 apples. I gave 2 apples to the neighbor and 2 to the repairman. I then went and bought 5 more apples and ate 1. How many apples did I remain with?
Let's calculate step by step:

1. You started with 10 apples.
2. You gave away 2 apples to the neighbor, so you have:
   \(10 - 2 = 8\) apples left.
3. Then you gave another 2 apples to the repairman, leaving you with:
   \(8 - 2 = 6\) apples.
4. After that, you bought 5 more apples, increasing your count to:
   \(6 + 5 = 11\) apples.
5. Finally, you ate 1 apple, which leaves you with:
   \(11 - 1 = 10\) apples.

So, you ended up with 10 apples remaining.


3. Structure the prompt to improve the result and ensure CoT prompting.

In [0]:
prompt="""
Give answer to the question using the problem statement. Let's think this step-by-step.
Problem Statement: I went to the market and bought 10 apples. I gave 2 apples to the neighbor and 2 to the repairman. I then went and bought 5 more apples and ate 1. 
Question: How many apples did I remain with?
Answer:
"""
result = pipe(prompt, min_new_tokens=5, max_new_tokens=200)
print(result[0]["generated_text"])


Give answer to the question using the problem statement. Let's think this step-by-step.
Problem Statement: I went to the market and bought 10 apples. I gave 2 apples to the neighbor and 2 to the repairman. I then went and bought 5 more apples and ate 1. 
Question: How many apples did I remain with?
Answer:
To find out how many apples you're left with, let's break down your purchases and expenditures:

1. You started with 10 apples.

2. You gave away 2 apples to the neighbor and another 2 to the repairman. So now you have:
   10 - 2 (neighbor) - 2 (repairman) = 6 apples remaining.

3. Then you bought 5 more apples, so you had:
   6 + 5 = 11 apples in total.

4. Finally, you ate 1 apple, leaving you with:
   11 - 1 = 10 apples.

Therefore, after all these transactions, you are left with 10 apples. The correct answer is D. 10. 

This process of subtraction from each purchase until reaching zero apples at the end gives us the final number of apples you ended up with. Each time you subtrac

####Use meta prompting along with CoT with a solution structure.

In [0]:
prompt="""
Give answer to the question using the problem statement. Let's think this step-by-step. 
Provide the answer using the given solution structure.
Solution Structure:
1. Begin with the response "Let's think this step-by-step"
2. Follow the reasoning steps, ensuring the solution is broken down clearly and logically.
3. End the solution with Final Answer
4. Finally state "The answer is [Final Answer to the problem]"
Problem Statement: I went to the market and bought 10 apples. I gave 2 apples to the neighbor and 2 to the repairman. I then went and bought 5 more apples and ate 1. 
Question: How many apples did I remain with?
Answer:
"""
result = pipe(prompt, min_new_tokens=5, max_new_tokens=200)
print(result[0]["generated_text"])


Give answer to the question using the problem statement. Let's think this step-by-step. 
Provide the answer using the given solution structure.
Solution Structure:
1. Begin with the response "Let's think this step-by-step"
2. Follow the reasoning steps, ensuring the solution is broken down clearly and logically.
3. End the solution with Final Answer
4. Finally state "The answer is [Final Answer to the problem]"
Problem Statement: I went to the market and bought 10 apples. I gave 2 apples to the neighbor and 2 to the repairman. I then went and bought 5 more apples and ate 1. 
Question: How many apples did I remain with?
Answer:
Let's think this step-by-step:

1. Initially, you had 10 apples.

2. You gave away 2 apples to the neighbor, leaving you with \(10 - 2 = 8\) apples.

3. Then, you gave another 2 apples to the repairman, leaving you with \(8 - 2 = 6\) apples.

4. Next, you bought 5 more apples, bringing your total to \(6 + 5 = 11\) apples.

5. After eating 1 apple, you were left 

Learn more about prompt engineering and different prompting techniques at https://www.promptingguide.ai